**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Databases

Every experiment you run produces data that outlives the script that made it. This workshop is about *keeping* that data queryable: relational modeling, SQL, and talking to a database from Python — using a sensor-logging scenario you'd actually meet in a signals lab. Everything runs on `sqlite3` from Python's standard library: zero installation.

## 0. Introduction

Why not CSV files? They rot: no types, no relationships, no protection against half-written rows, and every question becomes a bespoke parsing script. A relational database gives you **structure** (tables with types), **integrity** (constraints, transactions), and a **query language** (SQL) that answers questions you hadn't thought of when you designed the file.

## 1. Pre-requisites

- [Intro to Python](../../Intro_Func_Prog/Intro_Python/Intro_Python.ipynb) — functions, dicts, `with` blocks.
- Nothing else: `sqlite3` ships with Python.

---
### 🕐 Session 1 of 3 — *The Relational Model & SQL Basics* (~35 min)
**Goal:** design tables with keys and constraints; insert and query with SELECT/WHERE/ORDER BY.
**Feeds into:** Session 2 (joins & aggregation).

---

## 2. Tables, Keys, Constraints

💡 **Intuition.** A table is a *typed spreadsheet with rules*. The **primary key** gives every row an identity; a **foreign key** is a pointer from one table's rows to another's — the relational version of the C pointer from [Intro to C](../../Intro_Func_Prog/Intro_C.ipynb) §5, except the database *refuses* to let it dangle.

Our scenario: a lab logs signal recordings from multiple sensors. Two entities → two tables:

- `sensors` — one row per physical device.
- `recordings` — one row per capture, pointing at its sensor.

In [1]:
import sqlite3

db = sqlite3.connect(":memory:")          # RAM-only; use a filename to persist
db.execute("PRAGMA foreign_keys = ON")    # SQLite needs this opt-in!

db.executescript("""
CREATE TABLE sensors (
    sensor_id   INTEGER PRIMARY KEY,
    name        TEXT NOT NULL UNIQUE,
    kind        TEXT NOT NULL CHECK (kind IN ('accelerometer', 'microphone', 'ecg')),
    fs_hz       REAL NOT NULL CHECK (fs_hz > 0)
);
CREATE TABLE recordings (
    rec_id      INTEGER PRIMARY KEY,
    sensor_id   INTEGER NOT NULL REFERENCES sensors(sensor_id),
    started_at  TEXT NOT NULL,            -- ISO 8601 timestamp
    n_samples   INTEGER NOT NULL,
    rms         REAL                      -- a computed feature, maybe NULL until processed
);
""")
print("schema created")

schema created


In [2]:
db.executemany("INSERT INTO sensors (name, kind, fs_hz) VALUES (?, ?, ?)", [
    ("acc-01", "accelerometer", 1000.0),
    ("mic-01", "microphone",    48000.0),
    ("ecg-01", "ecg",           360.0),
])

db.executemany(
    "INSERT INTO recordings (sensor_id, started_at, n_samples, rms) VALUES (?, ?, ?, ?)", [
    (1, "2026-07-20T10:00:00", 60_000,  0.42),
    (1, "2026-07-20T11:00:00", 60_000,  0.51),
    (2, "2026-07-20T10:30:00", 480_000, 0.12),
    (2, "2026-07-21T09:00:00", 960_000, 0.19),
    (3, "2026-07-21T09:30:00", 21_600,  None),   # not yet processed
])
db.commit()
print("rows:", db.execute("SELECT COUNT(*) FROM recordings").fetchone()[0])

rows: 5


### 2.1. Constraints Earn Their Keep

Bad data is *rejected at the door* — a guarantee no CSV can make.

In [3]:
for bad in [
    ("INSERT INTO sensors (name, kind, fs_hz) VALUES ('x-01', 'thermometer', 10)", "bad kind"),
    ("INSERT INTO sensors (name, kind, fs_hz) VALUES ('acc-01', 'ecg', 360)",      "duplicate name"),
    ("INSERT INTO recordings (sensor_id, started_at, n_samples) VALUES (99, 't', 5)", "dangling foreign key"),
]:
    try:
        db.execute(bad[0])
    except sqlite3.IntegrityError as e:
        print(f"REJECTED ({bad[1]}): {e}")

REJECTED (bad kind): CHECK constraint failed: kind IN ('accelerometer', 'microphone', 'ecg')
REJECTED (duplicate name): UNIQUE constraint failed: sensors.name
REJECTED (dangling foreign key): FOREIGN KEY constraint failed


### 2.2. SELECT: Asking Questions

In [4]:
print("-- long recordings, newest first --")
for row in db.execute("""
    SELECT rec_id, sensor_id, started_at, n_samples
    FROM recordings
    WHERE n_samples >= 60000
    ORDER BY started_at DESC
"""):
    print(row)

print("-- unprocessed (rms IS NULL) --")
print(db.execute("SELECT rec_id FROM recordings WHERE rms IS NULL").fetchall())

-- long recordings, newest first --
(4, 2, '2026-07-21T09:00:00', 960000)
(2, 1, '2026-07-20T11:00:00', 60000)
(3, 2, '2026-07-20T10:30:00', 480000)
(1, 1, '2026-07-20T10:00:00', 60000)
-- unprocessed (rms IS NULL) --
[(5,)]


---
### 🕐 Session 2 of 3 — *Joins, Aggregation & Transactions* (~35 min)
**Goal:** combine tables with JOIN; summarize with GROUP BY; make multi-step changes atomic.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (databases from Python, files & analytics).

---

## 3. Joins & Aggregation

💡 **Intuition.** A **JOIN** re-follows the foreign-key pointers to reassemble the split entities: "for each recording, look up its sensor's row and staple them together." **GROUP BY** then folds rows sharing a key into one summary row — the SQL analogue of a NumPy `reduce` along an axis.

In [5]:
print("-- recordings with their sensor names (JOIN) --")
for row in db.execute("""
    SELECT r.rec_id, s.name, s.kind, r.n_samples / s.fs_hz AS seconds
    FROM recordings AS r
    JOIN sensors AS s ON s.sensor_id = r.sensor_id
"""):
    print(row)

-- recordings with their sensor names (JOIN) --
(1, 'acc-01', 'accelerometer', 60.0)
(2, 'acc-01', 'accelerometer', 60.0)
(3, 'mic-01', 'microphone', 10.0)
(4, 'mic-01', 'microphone', 20.0)
(5, 'ecg-01', 'ecg', 60.0)


In [6]:
print("-- per-sensor summary (GROUP BY) --")
for row in db.execute("""
    SELECT s.name,
           COUNT(*)            AS n_recs,
           SUM(r.n_samples)    AS total_samples,
           ROUND(AVG(r.rms), 3) AS mean_rms
    FROM recordings r
    JOIN sensors s USING (sensor_id)
    GROUP BY s.sensor_id
    HAVING COUNT(*) >= 1
    ORDER BY total_samples DESC
"""):
    print(row)

-- per-sensor summary (GROUP BY) --
('mic-01', 2, 1440000, 0.155)
('acc-01', 2, 120000, 0.465)
('ecg-01', 1, 21600, None)


Note `AVG(rms)` quietly *skipped* the NULL — aggregate functions ignore NULLs. Learn this before it surprises you in a paper's results table.

## 4. Transactions

💡 **Intuition.** A transaction makes several statements **all-or-nothing**. Classic example: moving a recording between sensors touches two rows; a crash between the two updates would corrupt the story. `BEGIN … COMMIT` means the database never shows the world a half-done state — and `ROLLBACK` is your undo.

In [7]:
try:
    with db:                                   # the connection as context manager = one transaction
        db.execute("UPDATE recordings SET sensor_id = 2 WHERE rec_id = 1")
        raise RuntimeError("power failure!")   # simulate a crash mid-transaction
except RuntimeError as e:
    print("crashed:", e)

# The update was rolled back automatically:
print("rec 1 still on sensor:", db.execute(
    "SELECT sensor_id FROM recordings WHERE rec_id = 1").fetchone()[0])

crashed: power failure!
rec 1 still on sensor: 1


---
### 🕐 Session 3 of 3 — *Databases from Python, Safely & at Scale* (~40 min)
**Goal:** parameterized queries, storing experiment results, and when to reach beyond SQLite.
**Builds on:** Session 2.

---

## 5. Python Patterns

### 5.1. Parameterized Queries — the Only Way

Never build SQL with f-strings: user input containing a quote breaks the query at best, *becomes* the query at worst (SQL injection). The `?` placeholder passes values out-of-band, so data can never be mistaken for code.

In [8]:
user_input = "acc-01'; DROP TABLE recordings; --"     # a hostile "sensor name"

# SAFE: value is bound as data — no row matches, nothing else happens
rows = db.execute("SELECT * FROM sensors WHERE name = ?", (user_input,)).fetchall()
print("safe query matched:", rows)
print("recordings table still exists:",
      db.execute("SELECT COUNT(*) FROM recordings").fetchone()[0], "rows")

safe query matched: []
recordings table still exists: 5 rows


### 5.2. A Reusable Results Logger

The pattern worth stealing: every training run / parameter sweep in your research logs into a table, and analysis is a query away — compare with juggling 40 `results_final_v2 (copy).csv` files.

In [9]:
db.executescript("""
CREATE TABLE IF NOT EXISTS experiments (
    exp_id     INTEGER PRIMARY KEY,
    ran_at     TEXT DEFAULT (datetime('now')),
    model      TEXT NOT NULL,
    lr         REAL NOT NULL,
    final_loss REAL NOT NULL
);
""")

def log_experiment(model, lr, final_loss):
    with db:
        db.execute("INSERT INTO experiments (model, lr, final_loss) VALUES (?, ?, ?)",
                   (model, lr, final_loss))

# pretend sweep — in real life these come from your training loop
for lr, loss in [(0.1, 0.083), (0.01, 0.041), (0.001, 0.067)]:
    log_experiment("mlp-32", lr, loss)

print("best run:", db.execute(
    "SELECT model, lr, final_loss FROM experiments ORDER BY final_loss LIMIT 1").fetchone())

best run: ('mlp-32', 0.01, 0.041)


### 5.3. Beyond SQLite

| Need | Reach for |
|---|---|
| Many concurrent writers, network access, roles | **PostgreSQL** — same SQL, industrial engine |
| Analytics over millions of rows / Parquet files | **DuckDB** — SQLite's analytical twin |
| Long dense signal data | Store *arrays* in files (HDF5/Parquet), *metadata + paths* in SQL — don't put 48 kHz samples one-per-row |

The SQL you wrote today transfers to all of them nearly verbatim.

## 6. Conclusion

You modeled entities as tables, protected them with constraints, asked real questions with joins and aggregation, made changes atomic, and built the experiment-logging habit. That's 90% of the database craft a signals/ML researcher needs.

---
## Where next

- [Intro to Operating Systems](../README.md#workshop-1--introduction-to-operating-systems-planned) — the files, processes, and locks a database is built from.
- [Intro to Python](../../Intro_Func_Prog/Intro_Python/Intro_Python.ipynb) — the NumPy side of the "arrays in files, metadata in SQL" pattern.